To begin, make a copy of this notebook to your own Google Drive: **File $\rightarrow$ Save a copy in Drive**.

### Submission Instructions:
1. **Download the notebook** (in the menubar, select File $\rightarrow$ Download .ipynb).
2. **Upload the downloaded notebook (.ipynb file) to your repository**.

Make sure you fill in any place that says `YOUR CODE HERE`, and all the cells include their outputs.

**Note:** To use a GPU, do the following: Runtime $\rightarrow$ Change runtime type $\rightarrow$ GPU.

In [ ]:
!pip install datasets
!pip install transformers
!pip install nltk
!pip install evaluate

In [ ]:
import nltk
nltk.download('punkt_tab')
## If you're using older version of nltk, you might need to use this line instead:
# nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import torch
import numpy as np

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

### Embeddings Setup

- **Direct Upload:** Click the **Files** icon on the left sidebar $\rightarrow$ **Upload** (temporary per session).
- **Google Drive via `gdown`:**


```python
import gdown
gdown.download('https://drive.google.com/uc?export=download&id=1TKNu5YltNk29orM0jm4gcYU3HKUygFSY', 'glove_vectors.npy', quiet=False)
gdown.download('https://drive.google.com/uc?export=download&id=14zqYRN_MFHwRCEBdzuWLhHLF6f-aDaqy', 'glove_vocab.json', quiet=False)
```

In [ ]:
import gdown
gdown.download('https://drive.google.com/uc?export=download&id=1TKNu5YltNk29orM0jm4gcYU3HKUygFSY', 'glove_vectors.npy', quiet=False)
gdown.download('https://drive.google.com/uc?export=download&id=14zqYRN_MFHwRCEBdzuWLhHLF6f-aDaqy', 'glove_vocab.json', quiet=False)

Downloading...
From (original): https://drive.google.com/uc?export=download&id=1TKNu5YltNk29orM0jm4gcYU3HKUygFSY
From (redirected): https://drive.google.com/uc?export=download&id=1TKNu5YltNk29orM0jm4gcYU3HKUygFSY&confirm=t&uuid=48944de1-3c61-4360-b244-f7f6a29c3010
To: /content/glove_vectors.npy
100%|██████████| 480M/480M [00:12<00:00, 37.4MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=14zqYRN_MFHwRCEBdzuWLhHLF6f-aDaqy
To: /content/glove_vocab.json
100%|██████████| 7.69M/7.69M [00:00<00:00, 124MB/s]


'glove_vocab.json'

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/imdb")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
EMBEDDINGS_PATH = "glove_vectors.npy"
VOCAB_PATH = "glove_vocab.json"

In [ ]:
from nltk.tokenize import word_tokenize
import json

with open(VOCAB_PATH) as f:
  vocab = json.load(f)

unk_idx = vocab["unk"]

def tokenize_function(example):
  tokenized_sentences = [word_tokenize(text.lower()) for text in example["text"]]
  tokenized_idx = [[vocab.get(token, unk_idx) for token in tokens] for tokens in tokenized_sentences]

  return {"labels":example['label'],'input_ids':tokenized_idx}

In [ ]:
train_dataset = raw_datasets['train'].shuffle(seed=42).map(tokenize_function,batched=True)
eval_dataset = raw_datasets['test'].shuffle(seed=42).map(tokenize_function,batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
def pad_sequence_to_length(
    sequence,
    desired_length: int,
    default_value = lambda: 0,
    padding_on_right: bool = True,
):
    sequence = list(sequence)
    # Truncates the sequence to the desired length.
    if padding_on_right:
        padded_sequence = sequence[:desired_length]
    else:
        padded_sequence = sequence[-desired_length:]
    # Continues to pad with default_value() until we reach the desired length.
    pad_length = desired_length - len(padded_sequence)
    # This just creates the default value once, so if it's a list, and if it gets mutated
    # later, it could cause subtle bugs. But the risk there is low, and this is much faster.
    values_to_pad = [default_value()] * pad_length
    if padding_on_right:
        padded_sequence = padded_sequence + values_to_pad
    else:
        padded_sequence = values_to_pad + padded_sequence
    return padded_sequence

In [ ]:
from evaluate import load
metric = load("accuracy")

### If you're using older versions, use:
# metric = dataset.load_metric("accuracy")

def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from dataclasses import dataclass

@dataclass
class DataCollatorWithPadding:

  def __call__(self, features):
    features_dict={}

    # Collect labels into a single tensor if present
    if "labels" in features[0]:
      features_dict["labels"] = torch.tensor([x["labels"] for x in features], dtype=torch.long)

    # Extract token sequences and determine the batch-level max sequence length
    input_ids = [x["input_ids"] for x in features]
    max_len = max(len(x) for x in input_ids)

    # Dynamically pad sequences to max_len
    features_dict["input_ids"] = torch.tensor([pad_sequence_to_length(x, max_len) for x in input_ids], dtype=torch.long)

    # Length-based attention mask: 1 for real tokens, 0 for padding
    masks = [[1] * len(x) for x in input_ids]
    features_dict["attention_masks"] = torch.tensor([pad_sequence_to_length(x, max_len) for x in masks], dtype=torch.long)

    return features_dict


## SECTION A - DAN implementation

Implement `DAN` **once**. Add arguments to `__init__` so Sections B–D can reuse the same class.

In [ ]:
from torch import nn

# Use nn.Sequential and nn.Linear for the network, and nn.CrossEntropyLoss for the loss.
# Make sure that the final layer has output dimension of size 2.
class DAN(nn.Module):
  def __init__(self, word_dropout=0.0, num_hidden_layers=2, hidden_dim=256, activation=nn.ReLU): # YOU ARE ALLOWED TO ADD ARGUMENTS TO THE __INIT__
          super().__init__()
          self.num_labels = 2
          # freeze=True keeps the pre-trained embeddings fixed during training (required).
          self.embeddings = nn.Embedding.from_pretrained(torch.FloatTensor(np.load(EMBEDDINGS_PATH)),
                                                         freeze=True)
          # YOUR CODE HERE
          self.word_dropout = word_dropout
          layers = []
          in_dim = self.embeddings.embedding_dim
          for _ in range(num_hidden_layers):
              layers.append(nn.Linear(in_dim, hidden_dim))
              layers.append(activation())
              in_dim = hidden_dim
          layers.append(nn.Linear(in_dim, self.num_labels))
          self.classifier = nn.Sequential(*layers)
          self.loss = nn.CrossEntropyLoss()
          # END YOUR CODE


  def forward(self, input_ids, attention_masks, labels=None, **kwargs):
      # YOUR CODE HERE
      emb = self.embeddings(input_ids)
      mask = attention_masks.float()
      if self.training and self.word_dropout > 0:
          keep = torch.bernoulli(torch.full_like(mask, 1 - self.word_dropout))
          mask = mask * keep
      mask = mask.unsqueeze(-1)
      avg = (emb * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
      # END YOUR CODE
      res = self.classifier(avg)
      loss = self.loss(res, labels)
      return {"loss": loss, "logits": res}


In [ ]:
# Define any TrainerCallback you need for the eval-accuracy-vs-epochs plot
# *before* creating the Trainer below.
# See: https://huggingface.co/transformers/main_classes/callback.html
from transformers import TrainerCallback

# YOUR CODE HERE
class EvaluationAccuracy(TrainerCallback):
    def __init__(self):
        self.accs = []
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        self.accs.append(metrics["eval_accuracy"])
# END YOUR CODE

In [ ]:
from transformers import Trainer, TrainingArguments

co = DataCollatorWithPadding()
training_args = TrainingArguments("DAN",
                                  # YOUR CODE HERE
                                  num_train_epochs=10, #must be at least 10.
                                  per_device_train_batch_size=32,
                                  per_device_eval_batch_size=32,
                                  learning_rate=1e-3,
                                  # END YOUR CODE

                                  save_total_limit=2,
                                  log_level="error",
                                  eval_strategy="epoch")
model = DAN()



trainer = Trainer(
    model=model,
    data_collator=co,
    args=training_args,
    callbacks = [
                 # YOUR CODE HERE
                 EvaluationAccuracy()
                 # END YOUR CODE
    ],
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.465819,0.406003,0.820360
2,0.406862,0.440965,0.791080
3,0.389929,0.378870,0.832120
4,0.386284,0.415343,0.812600


Epoch,Training Loss,Validation Loss,Accuracy
1,0.465819,0.406003,0.820360
2,0.406862,0.440965,0.791080
3,0.389929,0.378870,0.832120
4,0.386284,0.415343,0.812600
5,0.377857,0.431323,0.803640
6,0.373690,0.391297,0.827960


In [ ]:
# You may add Cells & Code lines if you need

In [ ]:
import matplotlib.pyplot as plt

eval_acc = [c for c in trainer.callback_handler.callbacks if isinstance(c, EvaluationAccuracy)][0]

plt.plot(range(1, len(eval_acc.accs) + 1), eval_acc.accs, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Eval accuracy")
plt.title("DAN eval accuracy per epoch")
plt.grid(True)
plt.show()

## SECTION B - Trying dropout

> **Read Section 3.1 of the paper carefully before implementing.**  

In [ ]:
# YOUR CODE HERE
dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
final_accs = []

for p in dropout_rates:
    torch.manual_seed(SEED)
    cb = EvaluationAccuracy()
    args = TrainingArguments(f"DAN_dropout_{p}",
                             num_train_epochs=10,
                             per_device_train_batch_size=32,
                             per_device_eval_batch_size=32,
                             learning_rate=1e-3,
                             save_strategy="no",
                             log_level="error",
                             eval_strategy="epoch")
    tr = Trainer(model=DAN(word_dropout=p),
                 data_collator=DataCollatorWithPadding(),
                 args=args,
                 callbacks=[cb],
                 train_dataset=train_dataset,
                 eval_dataset=eval_dataset,
                 compute_metrics=compute_metrics)
    tr.train()
    final_accs.append(cb.accs[-1])
    print(f"dropout={p}: final accuracy={cb.accs[-1]:.4f}")

plt.plot(dropout_rates, final_accs, marker="o")
plt.xlabel("Word dropout rate")
plt.ylabel("Eval accuracy (final epoch)")
plt.title("DAN accuracy vs. word dropout rate")
plt.grid(True)
plt.show()
# END YOUR CODE

## SECTION C - Different Layer numbers

In [ ]:
# YOUR CODE HERE
layer_counts = [0, 1, 2, 3]
layer_accs = []

for n in layer_counts:
    torch.manual_seed(SEED)
    cb = EvaluationAccuracy()
    args = TrainingArguments(f"DAN_layers_{n}",
                             num_train_epochs=10,
                             per_device_train_batch_size=32,
                             per_device_eval_batch_size=32,
                             learning_rate=1e-3,
                             save_strategy="no",
                             log_level="error",
                             eval_strategy="epoch")
    tr = Trainer(model=DAN(num_hidden_layers=n),
                 data_collator=DataCollatorWithPadding(),
                 args=args,
                 callbacks=[cb],
                 train_dataset=train_dataset,
                 eval_dataset=eval_dataset,
                 compute_metrics=compute_metrics)
    tr.train()
    layer_accs.append(cb.accs[-1])
    print(f"hidden layers={n}: final accuracy={cb.accs[-1]:.4f}")

plt.plot(layer_counts, layer_accs, marker="o")
plt.xticks(layer_counts)
plt.xlabel("Number of hidden layers")
plt.ylabel("Eval accuracy (final epoch)")
plt.title("DAN accuracy vs. number of hidden layers")
plt.grid(True)
plt.show()
# END YOUR CODE

## SECTION D - Activations

In [ ]:
# YOUR CODE HERE
activations = {"ReLU": nn.ReLU, "Tanh": nn.Tanh, "Sigmoid": nn.Sigmoid}
activation_histories = {}

for name, act in activations.items():
    torch.manual_seed(SEED)
    cb = EvaluationAccuracy()
    args = TrainingArguments(f"DAN_activation_{name}",
                             num_train_epochs=10,
                             per_device_train_batch_size=32,
                             per_device_eval_batch_size=32,
                             learning_rate=1e-3,
                             save_strategy="no",
                             log_level="error",
                             eval_strategy="epoch")
    tr = Trainer(model=DAN(activation=act),
                 data_collator=DataCollatorWithPadding(),
                 args=args,
                 callbacks=[cb],
                 train_dataset=train_dataset,
                 eval_dataset=eval_dataset,
                 compute_metrics=compute_metrics)
    tr.train()
    activation_histories[name] = cb.accs
    print(f"{name}: final accuracy={cb.accs[-1]:.4f}")

for name, accs in activation_histories.items():
    plt.plot(range(1, len(accs) + 1), accs, marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Eval accuracy")
plt.title("DAN accuracy across epochs by activation function")
plt.legend()
plt.grid(True)
plt.show()
# END YOUR CODE

## SECTION E - Sample 5 Misclassified Examples

Using your **top-performing** model from parts (a)–(d), find five movie reviews from the evaluation set that the model classified incorrectly. For each review, include a short explanation in your report describing why the model likely failed (See section 5.2 of the paper).

> `eval_dataset` was shuffled, so make sure you read the review text from the right source.

In [ ]:
# YOUR CODE HERE
from scipy.special import softmax

# Best model from sections (a)-(d): the configuration with the highest eval accuracy
# (84.01% in Section B). It is the Section A architecture (2 hidden layers of 256 units,
# ReLU, lr=1e-3, batch size 32, 10 epochs) with word dropout p=0.1.
torch.manual_seed(SEED)
best_cb = EvaluationAccuracy()
best_args = TrainingArguments("DAN_best",
                              num_train_epochs=10,
                              per_device_train_batch_size=32,
                              per_device_eval_batch_size=32,
                              learning_rate=1e-3,
                              save_strategy="no",
                              log_level="error",
                              eval_strategy="epoch")
best_trainer = Trainer(model=DAN(word_dropout=0.1),
                       data_collator=DataCollatorWithPadding(),
                       args=best_args,
                       callbacks=[best_cb],
                       train_dataset=train_dataset,
                       eval_dataset=eval_dataset,
                       compute_metrics=compute_metrics)
best_trainer.train()
print(best_cb.accs[-1])

# Find reviews the best model gets wrong, use Softmax for "probabilities" to see confidence
out = best_trainer.predict(eval_dataset)
preds = out.predictions.argmax(-1)
probs = softmax(out.predictions, axis=-1)
wrong = np.where(preds != out.label_ids)[0]
print(f"Misclassified: {len(wrong)} of {len(preds)}")

# View short misclassified ones that are easy to analyze.
short_wrong = [i for i in wrong if len(eval_dataset[int(i)]["text"]) < 700]
label_names = {0: "negative", 1: "positive"}

for i in short_wrong[:5]:
    i = int(i)
    print(f"--- eval_dataset index {i} ---")
    print("Review:", eval_dataset[i]["text"])
    print("True:", label_names[int(out.label_ids[i])],
          "| Predicted:", label_names[int(preds[i])],
          f"| Confidence: {probs[i].max():.2f}")
    print()
# END YOUR CODE

# SECTION F — Your embeddings vs. GloVe

Use your own Q2 word2vec embeddings, using the same architecture and the given hyperparameters. Bring in `word2vec_vectors.npy` and `word2vec_vocab.json` the same way as the GloVe files in Embeddings Setup, then run the cells below in order.

In [ ]:
EMBEDDINGS_PATH = "word2vec_vectors.npy"
VOCAB_PATH = "word2vec_vocab.json"

with open(VOCAB_PATH) as f:
    vocab = json.load(f)

unk_idx = vocab["unk"]

train_dataset = raw_datasets["train"].shuffle(seed=42).map(tokenize_function, batched=True)
eval_dataset = raw_datasets["test"].shuffle(seed=42).map(tokenize_function, batched=True)

In [ ]:
# Train a new model.
# YOUR CODE HERE
torch.manual_seed(SEED)
w2v_cb = EvaluationAccuracy()
w2v_args = TrainingArguments("DAN_word2vec",
                             num_train_epochs=15,
                             per_device_train_batch_size=32,
                             per_device_eval_batch_size=32,
                             learning_rate=2e-3,
                             save_strategy="no",
                             log_level="error",
                             eval_strategy="epoch")
w2v_trainer = Trainer(model=DAN(hidden_dim=32, num_hidden_layers=3),
                      data_collator=DataCollatorWithPadding(),
                      args=w2v_args,
                      callbacks=[w2v_cb],
                      train_dataset=train_dataset,
                      eval_dataset=eval_dataset,
                      compute_metrics=compute_metrics)
w2v_trainer.train()
print("Final accuracy:", w2v_cb.accs[-1])

plt.plot(range(1, len(w2v_cb.accs) + 1), w2v_cb.accs, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Eval accuracy")
plt.title("DAN with word2vec embeddings: accuracy per epoch")
plt.grid(True)
plt.show()
# END YOUR CODE